In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv("loan_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
loan_status=df['Loan_Status'].value_counts()
plt.title('Loan Status Distribution')
colors = ['pink', 'red'] 
plt.pie(loan_status.values,labels=loan_status.index,autopct='%1.1f%%',colors=colors)

In [ ]:
plt.subplots(figsize=(15, 5))

# Iterate over columns
for i, col in enumerate(['Gender', 'Married']):
    ax = plt.subplot(1, 2, i + 1)
    
    # Use 'Loan_Status' instead of 'Loan_status'
    sns.countplot(data=df, x=col, hue='Loan_Status', ax=ax)

    # Explicitly remove overlapping axes
#     ax.remove()

plt.tight_layout()
plt.show()


In [ ]:
plt.subplots(figsize=(15, 5))
for i, col in enumerate(['ApplicantIncome', 'LoanAmount']):
    plt.subplot(1, 2, i+1)
    sns.distplot(df[col])
plt.tight_layout()
plt.show()

In [ ]:
plt.subplots(figsize=(15, 5))
for i, col in enumerate(['ApplicantIncome', 'LoanAmount']):
    plt.subplot(1, 2, i+1)
    sns.boxplot(df[col])  #for finding outliers
    plt.xlabel(col)
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

# outliers present

In [ ]:
df = df[df['ApplicantIncome'] < 250000]
df = df[df['LoanAmount'] < 400000]


In [ ]:
pip install imblearn

In [ ]:
df['LoanAmount'].mean()

In [ ]:
df.groupby('Gender')['LoanAmount'].mean()

In [ ]:
df.groupby(['Married', 'Gender'])['LoanAmount'].mean()

# Observation is that married people take more loan than bachelor's 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn import metrics
from sklearn.svm import SVC
from imblearn.over_sampling import RandomOverSampler
# Function to apply label encoding
def encode_labels(data):
    for col in data.columns:
        if data[col].dtype == 'object':
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col])
 
    return data
 
# Applying function in whole column
df = encode_labels(df)
 
# Generating Heatmap
sns.heatmap(df.corr() > 0.8, annot=True, cbar=False)
plt.show()

In [ ]:
features = df.drop('Loan_Status', axis=1)
target = df['Loan_Status'].values
 
X_train, X_val,\
    Y_train, Y_val = train_test_split(features, target,
                                      test_size=0.2,
                                      random_state=10)
 
# As the data was highly imbalanced we will balance
# it by adding repetitive rows of minority class.
ros = RandomOverSampler(sampling_strategy='minority',
                        random_state=0)
X, Y = ros.fit_resample(X_train, Y_train)
 
X_train.shape, X.shape

In [ ]:
# Normalizing the features for stable and fast training.
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_val = scaler.transform(X_val)

In [ ]:
from sklearn.metrics import roc_auc_score
model = SVC(kernel='rbf')
model.fit(X, Y)
 
print('Training Accuracy : ', metrics.roc_auc_score(Y, model.predict(X)))
print('Validation Accuracy : ', metrics.roc_auc_score(Y_val, model.predict(X_val)))
print()

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
training_roc_auc = roc_auc_score(Y, model.predict(X))
validation_roc_auc = roc_auc_score(Y_val, model.predict(X_val))
print('Training ROC AUC Score:', training_roc_auc)
print('Validation ROC AUC Score:', validation_roc_auc)
print()
cm = confusion_matrix(Y_val, model.predict(X_val))

In [ ]:
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(Y_val, model.predict(X_val)))